# 03 — Building an Eval Harness

## Why this notebook exists

In **`02_assertions_and_golden_outputs.ipynb`** we built five grader functions — exact match, contains, regex, schema validation, golden outputs — each following the same shape: take an example and an agent output, return a score. That was a useful pattern, but we applied each grader by hand, one call at a time, against a single example. When you have a dataset of dozens or hundreds of examples and several graders to apply, doing that by hand doesn't scale.

This notebook generalizes the pattern into a **reusable eval harness**: a small set of dataclasses that model the inputs (`Example`), the per-example results (`ExampleResult`), and the aggregated report (`EvalReport`), plus a `run_eval(agent, dataset, graders)` function that wires them together. By the end you'll be able to add a new grader or a new dataset and re-run your entire eval suite in one call.

No API key needed — the agent is a deterministic Python stub and all graders run in-process.

## What you'll learn

- How to model an evaluation as three composable pieces: an `Example` (input + expected), an `ExampleResult` (example + output + scores), and an `EvalReport` (aggregated results with metrics).
- How to write `run_eval(agent, dataset, graders) -> EvalReport` — a runner that applies every grader to every example and collects structured results.
- How to read an `EvalReport`: `pass_rate()`, `mean_score()`, per-grader breakdowns, and `summary_table()`.
- Why pass rate and mean score tell different stories about the same run.
- A small but real Gotcha: graders from notebook 02 read `example["expected"]` (dict form). Now that `Example` is a dataclass, they must read `example.expected` (attribute form). The graders in this notebook are re-declared to match.

## 1. Setup

We re-declare `Score` and every deterministic grader from notebook 02 here, so this notebook is fully self-contained and runs in a fresh kernel without importing from another file.

**One small change from notebook 02:** the graders there read `example["expected"]` because examples were plain dicts. In this notebook, `Example` is a dataclass, so the graders read `example.expected` (attribute access). Everything else is identical — same function names, same signatures, same `Score` return type.

> **Gotcha:** If you copy a grader verbatim from notebook 02 and forget to change `example["expected"]` to `example.expected`, you'll get a `TypeError: 'Example' object is not subscriptable`. The fix is one character: replace `[` with `.` and drop the `"]"`. This is the only breaking change between the two notebooks.

In [ ]:
from __future__ import annotations

import re
import json
from dataclasses import dataclass, field
from typing import Any, Callable

from pydantic import BaseModel, ValidationError


# ---------------------------------------------------------------------------
# Score — the atomic unit every grader returns
# ---------------------------------------------------------------------------

@dataclass
class Score:
    key: str          # grader identity, e.g. "exact_match"
    score: float      # numeric value in [0, 1]
    passed: bool      # True if this grader considers the output acceptable
    comment: str = "" # optional human-readable note


# ---------------------------------------------------------------------------
# Deterministic graders — re-declared from notebook 02, now reading
# example.expected (attribute form) instead of example["expected"] (dict form)
# ---------------------------------------------------------------------------

def exact_match(example, output) -> Score:
    """Pass if str(output) == str(example.expected), case-insensitive strip."""
    expected = str(example.expected).strip().lower()
    actual = str(output).strip().lower()
    passed = expected == actual
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment=f"expected={expected!r}, got={actual!r}",
    )


def make_contains(substring: str, key: str = "contains") -> Callable:
    """Return a grader that passes when `substring` appears in the output."""
    def grader(example, output) -> Score:
        passed = substring.lower() in str(output).lower()
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"looked for {substring!r}",
        )
    return grader


def make_regex(pattern: str, key: str = "regex") -> Callable:
    """Return a grader that passes when `pattern` matches anywhere in the output."""
    compiled = re.compile(pattern, re.IGNORECASE)
    def grader(example, output) -> Score:
        passed = bool(compiled.search(str(output)))
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"pattern={pattern!r}",
        )
    return grader


def make_schema_grader(model: type[BaseModel], key: str = "valid_schema") -> Callable:
    """Return a grader that passes when the output parses into `model`."""
    def grader(example, output) -> Score:
        try:
            if isinstance(output, str):
                data = json.loads(output)
            else:
                data = output
            model.model_validate(data)
            return Score(key=key, score=1.0, passed=True, comment="schema valid")
        except (ValidationError, json.JSONDecodeError, TypeError) as exc:
            return Score(key=key, score=0.0, passed=False, comment=str(exc))
    return grader


def make_golden_grader(path: str, key: str = "golden") -> Callable:
    """Return a grader that passes when output matches the text stored at `path`."""
    import pathlib
    golden = pathlib.Path(path).read_text().strip()
    def grader(example, output) -> Score:
        actual = str(output).strip()
        passed = actual == golden
        return Score(
            key=key,
            score=1.0 if passed else 0.0,
            passed=passed,
            comment=f"golden={golden[:40]!r}",
        )
    return grader


print("Setup OK — Score and graders declared (attribute form)")

## 2. The Data Model

An **example** pairs an `input` (a dict the agent receives) with an optional `expected` value (whatever the correct output should be). A `metadata` dict holds anything else — source, difficulty tier, tags — that we might want to filter on later without changing the core fields.

```python
@dataclass
class Example:
    input: dict
    expected: object | None = None
    metadata: dict = field(default_factory=dict)
```

A **dataset** is just a `list[Example]`. No framework required — it's a Python list you can build inline, load from a JSON file, or pull from a database.

We'll use a small sentiment-classification dataset: five product reviews with expected labels (`"positive"`, `"negative"`, or `"neutral"`). One example is deliberately tricky (a sarcastic review) so the stub agent we build in the next section gets it wrong — making the metrics more interesting than a trivial 100 % pass rate.

In [ ]:
# ---------------------------------------------------------------------------
# Example dataclass
# ---------------------------------------------------------------------------

@dataclass
class Example:
    input: dict
    expected: object | None = None
    metadata: dict = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Dataset: 5 sentiment-classification cases
# ---------------------------------------------------------------------------

dataset: list[Example] = [
    Example(
        input={"review": "This product is absolutely fantastic! Best purchase I've made this year."},
        expected="positive",
        metadata={"id": "s01", "difficulty": "easy"},
    ),
    Example(
        input={"review": "Terrible quality. Broke after two days and support never responded."},
        expected="negative",
        metadata={"id": "s02", "difficulty": "easy"},
    ),
    Example(
        input={"review": "It arrived on time and the packaging was fine."},
        expected="neutral",
        metadata={"id": "s03", "difficulty": "medium"},
    ),
    Example(
        input={"review": "Oh great, another product that stops working the moment the warranty expires."},
        expected="negative",
        metadata={"id": "s04", "difficulty": "hard", "note": "sarcastic — stub gets this wrong"},
    ),
    Example(
        input={"review": "Exactly what I needed. Simple, effective, good value."},
        expected="positive",
        metadata={"id": "s05", "difficulty": "easy"},
    ),
]

print(f"Dataset: {len(dataset)} examples")
for ex in dataset:
    preview = ex.input["review"][:55].replace("\n", " ")
    print(f"  [{ex.metadata['id']}] expected={ex.expected!r:10s}  {preview!r}")

## 3. A Toy Agent to Evaluate

A real sentiment classifier would call an LLM. For this notebook we use a deterministic stub: a function that classifies reviews using keyword matching. It's correct on the four straightforward examples but returns `"positive"` for the sarcastic review (`s04`), which should be `"negative"`.

That one wrong answer is intentional — if the stub were perfect, every metric would be 100 % and we'd learn nothing about reading a report.

The harness doesn't care how the agent works internally. It only requires that `classify_agent(input: dict) -> str` — it takes the same `input` dict that `Example.input` holds and returns a label string.

In [ ]:
def classify_agent(input: dict) -> str:
    """Deterministic stub sentiment classifier using keyword matching.

    Correct on s01, s02, s03, s05.
    Intentionally wrong on s04 (sarcastic review) — returns 'positive'.
    """
    review = input["review"].lower()

    negative_words = {"terrible", "broke", "awful", "horrible", "worst", "bad", "poor"}
    positive_words = {"fantastic", "great", "excellent", "best", "needed", "effective"}
    neutral_words  = {"arrived", "time", "packaging", "fine"}

    neg_hits = sum(1 for w in negative_words if w in review)
    pos_hits = sum(1 for w in positive_words if w in review)
    neu_hits = sum(1 for w in neutral_words  if w in review)

    if neg_hits > pos_hits and neg_hits > neu_hits:
        return "negative"
    elif neu_hits > pos_hits and neu_hits >= neg_hits:
        return "neutral"
    else:
        return "positive"


# Quick smoke test so we can see the one wrong prediction before running the harness
print("Smoke test — classify_agent predictions:")
for ex in dataset:
    prediction = classify_agent(ex.input)
    correct = "✓" if prediction == ex.expected else "✗"
    print(f"  [{ex.metadata['id']}] {correct} predicted={prediction!r:10s} expected={ex.expected!r}")

## 4. The Runner

Three more pieces complete the harness:

- **`ExampleResult`** — holds one example, the agent's output for it, and the list of `Score`s every grader produced.
- **`EvalReport`** — wraps the full list of `ExampleResult`s and exposes three read methods: `pass_rate(key=None)`, `mean_score(key=None)`, and `summary_table()`.
- **`run_eval(agent, dataset, graders) -> EvalReport`** — loops over the dataset, calls `agent(example.input)` for each, applies every grader, and assembles the report.

The function signature `run_eval(agent, dataset, graders)` is the core abstraction of the series: you plug in any callable agent, any list of examples, and any list of grader functions, and you get back a structured report. Changing the agent, dataset, or grader list is one argument swap.

In [ ]:
# ---------------------------------------------------------------------------
# ExampleResult and EvalReport
# ---------------------------------------------------------------------------

@dataclass
class ExampleResult:
    example: Example
    output: object
    scores: list  # list[Score]


@dataclass
class EvalReport:
    results: list  # list[ExampleResult]

    def _scores(self, key=None):
        """Yield all Score objects, optionally filtered to a single grader key."""
        for r in self.results:
            for s in r.scores:
                if key is None or s.key == key:
                    yield s

    def pass_rate(self, key=None) -> float:
        """Fraction of scores (optionally filtered to grader `key`) with passed=True."""
        scores = list(self._scores(key))
        if not scores:
            return 0.0
        return sum(1 for s in scores if s.passed) / len(scores)

    def mean_score(self, key=None) -> float:
        """Mean of .score values (optionally filtered to grader `key`)."""
        scores = list(self._scores(key))
        if not scores:
            return 0.0
        return sum(s.score for s in scores) / len(scores)

    def summary_table(self) -> str:
        """Printable text table: one row per grader key with pass_rate + mean_score."""
        # Collect unique grader keys in insertion order
        seen: dict[str, None] = {}
        for r in self.results:
            for s in r.scores:
                seen[s.key] = None
        keys = list(seen)

        header = f"{'Grader':<20}  {'Pass rate':>10}  {'Mean score':>10}"
        sep    = "-" * len(header)
        rows   = [header, sep]
        for k in keys:
            pr = self.pass_rate(k)
            ms = self.mean_score(k)
            rows.append(f"{k:<20}  {pr:>10.1%}  {ms:>10.3f}")
        rows.append(sep)
        overall_pr = self.pass_rate()
        overall_ms = self.mean_score()
        rows.append(f"{'OVERALL':<20}  {overall_pr:>10.1%}  {overall_ms:>10.3f}")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# run_eval
# ---------------------------------------------------------------------------

def run_eval(agent, dataset: list, graders: list) -> EvalReport:
    """Run every example in `dataset` through `agent`, apply every grader.

    Args:
        agent:   callable (input: dict) -> str | dict
        dataset: list[Example]
        graders: list of grader callables, each (example, output) -> Score

    Returns:
        EvalReport containing one ExampleResult per example.
    """
    results = []
    for example in dataset:
        output = agent(example.input)
        scores = [g(example, output) for g in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results=results)

In [ ]:
# Grader set for this run
graders = [
    exact_match,
    make_contains("positive", key="contains_positive"),
]

report = run_eval(
    agent=classify_agent,
    dataset=dataset,
    graders=graders,
)

print(f"EvalReport produced: {len(report.results)} ExampleResult(s), "
      f"{sum(len(r.scores) for r in report.results)} Score(s) total")

## 5. Reading the Report

An `EvalReport` exposes three views of the same underlying data:

- `summary_table()` — one row per grader, side by side: a quick dashboard.
- `pass_rate(key=None)` — what fraction of outputs satisfied the pass criterion. When called with a grader key, scoped to that grader only.
- `mean_score(key=None)` — the numeric average of `Score.score` values. For binary graders (0/1) this equals pass rate, but graders that return partial scores (e.g. 0.5 for a fuzzy match) show a different picture here.

A per-example breakdown loop lets you see *which* examples failed — essential for debugging the agent rather than just knowing a single aggregate number.

In [ ]:
# Summary table
print("=== Summary table ===")
print(report.summary_table())
print()

# Aggregate metrics
print(f"Overall  pass_rate : {report.pass_rate():.1%}")
print(f"exact_match  pass_rate : {report.pass_rate('exact_match'):.1%}")
print(f"contains_positive  pass_rate : {report.pass_rate('contains_positive'):.1%}")
print(f"Overall  mean_score: {report.mean_score():.3f}")
print()

# Per-example breakdown
print("=== Per-example breakdown ===")
for r in report.results:
    ex_id = r.example.metadata.get("id", "?")
    label_line = f"[{ex_id}] output={r.output!r:12s} expected={r.example.expected!r}"
    score_parts = []
    for s in r.scores:
        mark = "✓" if s.passed else "✗"
        score_parts.append(f"{s.key}={mark}")
    print(f"  {label_line}  |  {', '.join(score_parts)}")

print()
print("Pass rate tells you the fraction of outputs that cleared the bar.")
print("Mean score is the same number for binary graders, but differs once")
print("graders return partial credit (e.g. a fuzzy similarity score in [0, 1]).")

### Try it

Swap in a buggier agent — one that always returns `"positive"` regardless of the review — and re-run the harness with the same grader set. Watch how `exact_match` pass rate drops from 80 % to 40 % and `contains_positive` jumps to 100 %. This is the key insight: metrics only mean something in the context of *which graders you chose* and *what the agent is supposed to do*.

Try changing `graders` to include only `exact_match` and re-run, or add a `make_contains("negative", key="contains_negative")` grader and observe that the broken agent scores 0 % on it.

In [ ]:
def always_positive_agent(input: dict) -> str:
    """Broken agent: always returns 'positive', ignores the review."""
    return "positive"


bugged_report = run_eval(
    agent=always_positive_agent,
    dataset=dataset,
    graders=[
        exact_match,
        make_contains("positive", key="contains_positive"),
        make_contains("negative", key="contains_negative"),
    ],
)

print("=== Bugged agent — summary table ===")
print(bugged_report.summary_table())
print()
print("Compare to the original agent above:")
print(f"  exact_match pass_rate: original={report.pass_rate('exact_match'):.0%}  "
      f"bugged={bugged_report.pass_rate('exact_match'):.0%}")
print(f"  contains_positive pass_rate: original={report.pass_rate('contains_positive'):.0%}  "
      f"bugged={bugged_report.pass_rate('contains_positive'):.0%}")

## What you just learned

- An eval is three composable pieces: `Example` (input + expected), `ExampleResult` (example + output + scores), and `EvalReport` (aggregated results with metrics). Each is a plain Python dataclass — no framework needed.
- `run_eval(agent, dataset, graders) -> EvalReport` is the central abstraction: swap any one of the three arguments and re-run. The graders, the agent, and the dataset are all independently replaceable.
- Graders from notebook 02 carry over verbatim except for one small change: `example["expected"]` (dict form) becomes `example.expected` (attribute form). Same logic, same `Score` return type.
- `pass_rate()` and `mean_score()` measure the same underlying data through two different lenses. For binary graders they're equal; once graders assign partial credit they diverge — and that divergence is informative.
- A per-example breakdown loop is often more useful than aggregate metrics alone: it tells you *which* examples failed and *why*, making the harness a debugging tool, not just a scoreboard.

## What's missing

We now have a harness that can evaluate a single agent snapshot against a fixed dataset. But agents evolve — you fix a bug, change a prompt, update a dependency — and what was passing yesterday might silently break today.

In **`04_regression_testing.ipynb`** we extend this harness into a **regression gate**: treat the eval dataset as a versioned artifact, run the harness against two agent versions (v1 and a deliberately-regressed v2), diff the scores, and define a gate that fails if any example regresses or if aggregate quality drops below a threshold. Same `run_eval` abstraction, same graders, same `EvalReport` — now used to *compare* rather than just *measure*.